<a href="https://colab.research.google.com/github/deekaykay07-hub/kali-wireless-bridge/blob/main/tui-unrestricted.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kali-Wireless TUI — Remote 15B Unrestricted Model (Colab T4 + ngrok)

**Model:** `yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF`

This notebook runs the strong **Unrestricted 15B** model on Colab T4 and exposes it via ngrok so your Kali TUI (on the droplet) can use it as a remote backend.

### Instructions
1. Runtime → Change runtime type → **T4 GPU** (High-RAM recommended)
2. Add your ngrok token as Colab Secret named `NGROK_AUTH_TOKEN`
3. Run cells in order
4. Copy the final ngrok URL and put it in your `docker-compose.remote-colab.yml`

In [1]:
#@title 1. Install Ollama + dependencies
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw 2>/dev/null || true
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q huggingface_hub pyngrok
print("[+] Ollama + dependencies installed")
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Selecting previously unselected package usb.ids.
Prepari

In [2]:
#@title 2. Start Ollama server (Manual Background Process)
import subprocess, os, time, requests

# 1. Kill any existing instances
!pkill -f 'ollama serve' || true
time.sleep(2)

# 2. Configure environment for GPU and Debugging
env = os.environ.copy()
env.update({
    "OLLAMA_HOST": "0.0.0.0:11434",
    "OLLAMA_FLASH_ATTENTION": "1",
    "CUDA_VISIBLE_DEVICES": "0",
    "OLLAMA_DEBUG": "1"
})

print("[*] Starting Ollama server via background subprocess...")
# 3. Start server and pipe output to a log file instead of systemd
with open('ollama_log.txt', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=log_file, stderr=log_file)

# 4. Polling loop to wait for the API to become ready
print("[*] Waiting for server to initialize...")
for i in range(30):
    try:
        res = requests.get('http://localhost:11434/api/tags', timeout=2)
        if res.status_code == 200:
            print("[+] Ollama is running and responsive on port 11434.")
            break
    except:
        pass
    if i % 5 == 0 and i > 0: print(f"    ...waiting ({i}s)")
    time.sleep(1)
else:
    print("[!] CRITICAL: Ollama failed to start. Check 'ollama_log.txt' for errors.")

# 5. Verify GPU status in the log
!grep -i "nvidia" ollama_log.txt || echo "[!] GPU not mentioned in logs yet."

^C
[*] Starting Ollama server via background subprocess...
[*] Waiting for server to initialize...
[+] Ollama is running and responsive on port 11434.
time=2026-06-01T21:30:03.500Z level=DEBUG source=server.go:434 msg=subprocess CUDA_VERSION=12.8.1 LD_LIBRARY_PATH=/usr/local/lib/ollama:/usr/local/lib/ollama/cuda_v12:/usr/lib64-nvidia PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin OLLAMA_HOST=0.0.0.0:11434 OLLAMA_FLASH_ATTENTION=1 CUDA_VISIBLE_DEVICES=0 OLLAMA_DEBUG=1 OLLAMA_LIBRARY_PATH=/usr/local/lib/ollama:/usr/local/lib/ollama/cuda_v12
time=2026-06-01T21:30:03.789Z level=DEBUG source=server.go:434 msg=subprocess CUDA_VERSION=12.8.1 LD_LIBRARY_PATH=/usr/local/lib/ollama:/usr/local/lib/ollama/cuda_v13:/usr/lib64-nvidia PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin OLLAMA_HOST=0.0.0.0:11434 OLLAMA_FLASH_

In [3]:
#@title 3. Download GGUF + Create Model with Exact Name
from huggingface_hub import hf_hub_download
import subprocess, os

repo_id = "yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF"
filename = "unrestricted-knowledge-will-not-refuse-15b-q4_k_m.gguf"
DESIRED_NAME = "yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF"

print(f"[*] Downloading {filename}... (this will take a while)")
model_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    cache_dir="/content/model_download"
)
print(f"[+] Downloaded to {model_path}")

print(f"[*] Creating Modelfile...")
with open("Modelfile", "w") as f:
    f.write(f"FROM {model_path}\n")

print(f"[*] Creating model as: {DESIRED_NAME}")
!ollama create {DESIRED_NAME} -f Modelfile

print("\n[+] Current models:")
!ollama list

print(f"\n[*] Warming up {DESIRED_NAME}...")
!ollama run {DESIRED_NAME} "System ready." --verbose 2>&1 | tail -5

[*] Downloading unrestricted-knowledge-will-not-refuse-15b-q4_k_m.gguf... (this will take a while)


unrestricted-knowledge-will-not-refuse-1(…):   0%|          | 0.00/8.96G [00:00<?, ?B/s]

[+] Downloaded to /content/model_download/models--yqkm--Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF/snapshots/289e6f79c509538fd5050e46b49ab56ea014e5c8/unrestricted-knowledge-will-not-refuse-15b-q4_k_m.gguf
[*] Creating Modelfile...
[*] Creating model as: yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF


[+] Current models:
NAME                                                                  ID              SIZE      MODIFIED               
yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF:latest    39d5d28f4f93    9.0 GB    Less than a second ago    

[*] Warming up yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF...
prompt eval rate:     0.28 tokens/s
eval count:           66 token(s)
eval duration:        3.939242268s
eval rate:            16.75 tokens/s


In [4]:
#@title 4. Start ngrok
from google.colab import userdata
import subprocess, time, requests

NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
!ngrok config add-authtoken {NGROK_TOKEN}
!pkill -f ngrok || true
time.sleep(1)

print("[*] Starting ngrok...")
subprocess.Popen(['ngrok', 'http', '11434'])
time.sleep(7)

r = requests.get('http://localhost:4040/api/tunnels', timeout=10)
url = [t['public_url'] for t in r.json()['tunnels'] if t['proto'] == 'https'][0]

print("\n" + "="*70)
print("✅ SUCCESS")
print("="*70)
print(f"\nPUBLIC OLLAMA URL: {url}")
print(f"\nUse these in docker-compose.remote-colab.yml:")
print(f"  OLLAMA_HOST={url}")
print(f"  MISTRAL_MODEL=yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF")
print("="*70)

with open('/content/remote_ollama_url.txt', 'w') as f:
    f.write(url)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
^C
[*] Starting ngrok...

✅ SUCCESS

PUBLIC OLLAMA URL: https://pretense-casualty-copier.ngrok-free.dev

Use these in docker-compose.remote-colab.yml:
  OLLAMA_HOST=https://pretense-casualty-copier.ngrok-free.dev
  MISTRAL_MODEL=yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF


In [5]:
#@title 5. Keep-alive (run this in a separate cell and leave it running)
import time, requests
from datetime import datetime

print("🔄 Keep-alive running. This prevents Colab from disconnecting.")
print("Stop this cell when you are done.")

while True:
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
        print(f"[{datetime.now().strftime('%H:%M')}] Still alive")
    except:
        pass
    time.sleep(300)

🔄 Keep-alive running. This prevents Colab from disconnecting.
Stop this cell when you are done.
[21:36] Still alive
[21:41] Still alive
[21:46] Still alive
[21:51] Still alive
[21:56] Still alive
[22:01] Still alive
[22:06] Still alive
[22:11] Still alive
[22:16] Still alive
[22:21] Still alive
[22:26] Still alive
[22:31] Still alive


KeyboardInterrupt: 

## Final Steps

1. Copy the **PUBLIC OLLAMA URL** printed above.
2. On your droplet:
   ```bash
   cd /root/kali-mistral-tui
   nano docker-compose.remote-colab.yml
   ```
3. Set:
   ```yaml
   - OLLAMA_HOST=https://your-ngrok-url.ngrok-free.dev
   - MISTRAL_MODEL=yqkm/Unrestricted-Knowledge-Will-Not-Refuse-15B-Q4_K_M-GGUF
   ```
4. Restart the TUI:
   ```bash
   docker compose -f docker-compose.remote-colab.yml down
   docker compose -f docker-compose.remote-colab.yml up -d
   ```

**Note:** The model is heavy (15B). If you get OOM, you may need to fall back to a smaller model or use the 7B version.